# ECE 214B Project 2: Final Colab Run

This notebook is a runner only. All scientific/model logic lives in `project/scripts/` and `project/src/`.

Workflow: upload `colab_drop_final/` to Google Drive, rename it to `214B_colab_drop_final`, keep the dataset zip separately in Drive, edit `DATA_ZIP`, set a GPU runtime, then run all cells. The dataset zip is extracted to local Colab storage before experiments run.


## Setup

Edit only these paths. `DATA_ZIP` is the dataset zip already stored separately in Google Drive.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

DROP_DIR = "/content/drive/MyDrive/214B_colab_drop_final"
DATA_ZIP = "/content/drive/MyDrive/path/to/S26_ECE_214B_Mini_Project_2.zip"
EXTRACT_DIR = "/content/214B_data"

PROJECT_DIR = f"{DROP_DIR}/project"
OUTPUT_ZIP = f"{DROP_DIR}/outputs/outputs_colab_final.zip"

print("DROP_DIR:", DROP_DIR)
print("DATA_ZIP:", DATA_ZIP)
print("EXTRACT_DIR:", EXTRACT_DIR)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

drop_path = Path(DROP_DIR)
project_path = Path(PROJECT_DIR)
data_zip_path = Path(DATA_ZIP)

if not drop_path.exists():
    raise FileNotFoundError(f"DROP_DIR not found: {drop_path}")
if not project_path.exists():
    raise FileNotFoundError(f"PROJECT_DIR not found: {project_path}")
for required in [project_path / "scripts", project_path / "src", project_path / "requirements.txt"]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required project item: {required}")
if not data_zip_path.exists():
    raise FileNotFoundError(f"DATA_ZIP not found. Edit DATA_ZIP near the top of the notebook: {data_zip_path}")

Path(f"{DROP_DIR}/outputs").mkdir(parents=True, exist_ok=True)
print("Drive/drop validation passed.")


## Dataset Extraction

The zip is extracted into local Colab storage so experiments do not stream WAV files from Drive.


In [ ]:
extract_path = Path(EXTRACT_DIR)
if extract_path.exists():
    shutil.rmtree(extract_path)
extract_path.mkdir(parents=True, exist_ok=True)

print(f"Extracting {DATA_ZIP} -> {EXTRACT_DIR}")
with zipfile.ZipFile(DATA_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def find_dataset_dir(root: Path) -> Path:
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for candidate in candidates:
        if all((candidate / name).exists() for name in ["splits", "sessions", "wavs"]):
            return candidate
    raise FileNotFoundError(f"Could not find extracted dataset folder containing splits/, sessions/, wavs/ under {root}")

DATA_DIR = str(find_dataset_dir(extract_path))
print("DATA_DIR:", DATA_DIR)


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import platform

print("Python:", sys.version)
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: No GPU is available. CPU experiments can run, but transformer/acoustic embedding experiments may be slow.")
except Exception as exc:
    print("WARNING: torch is not importable yet:", repr(exc))
print("Platform:", platform.platform())


## Core Baselines


In [ ]:
core_commands = [
    [sys.executable, "scripts/audit_dataset.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_descriptor_baselines.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_acoustic_descriptor_baseline.py", "--data_dir", DATA_DIR],
]
for command in core_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Transformer Baselines

These use frozen pretrained embeddings. HuBERT is final layer/layer 12 only here; the full HuBERT layer sweep is intentionally not part of the default run-all path.


In [ ]:
transformer_commands = [
    [sys.executable, "scripts/run_roberta_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_hubert_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_wavlm_baseline.py", "--data_dir", DATA_DIR],
]
for command in transformer_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Fusion And Thresholding


In [ ]:
fusion_threshold_commands = [
    [sys.executable, "scripts/run_fusion.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_clinical_thresholding.py", "--data_dir", DATA_DIR],
]
for command in fusion_threshold_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Phase 2 Interpretability


In [ ]:
phase2_commands = [
    [sys.executable, "scripts/run_linguistic_marker_panel.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/analyze_marker_errors.py", "--data_dir", DATA_DIR],
]
for command in phase2_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Phase 3 Hybrid And Clinical Analysis


In [ ]:
phase3_commands = [
    [sys.executable, "scripts/run_uncertainty_gated_marker_fusion.py"],
    [sys.executable, "scripts/run_marker_error_corrector.py"],
    [sys.executable, "scripts/run_stacked_tfidf_marker_meta_model.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_marker_stratified_thresholding.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_session_aggregation_ablation.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_age_aware_thresholding.py", "--data_dir", DATA_DIR],
]
for command in phase3_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Final Results Collection


In [ ]:
subprocess.run([sys.executable, "scripts/collect_required_results.py"], check=True)


## Final Output Packaging


In [ ]:
output_zip = Path(OUTPUT_ZIP)
output_zip.parent.mkdir(parents=True, exist_ok=True)
if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(str(output_zip.with_suffix("")), "zip", root_dir=PROJECT_DIR, base_dir="outputs")

# Add reports/ into the same zip if present.
import zipfile
reports_dir = Path(PROJECT_DIR) / "reports"
if reports_dir.exists():
    with zipfile.ZipFile(output_zip, "a", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in reports_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(PROJECT_DIR))

print("Final output zip:", output_zip)
print("Size MB:", round(output_zip.stat().st_size / (1024 * 1024), 2))


## Optional: Whisper ASR-Error Features

Manual exploratory experiment. This may take time and is not run by default. It uses local/open-source Whisper in Colab and caches transcripts in `outputs/asr_error_features/`.


In [ ]:
# Optional manual run: Whisper ASR-error features.
# First try a small run; remove --max_sessions for the full experiment.
# subprocess.run([sys.executable, "scripts/run_asr_error_features.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/asr_error_features", "--whisper_model", "base", "--max_sessions", "10"], check=True)
# subprocess.run([sys.executable, "scripts/run_asr_error_features.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/asr_error_features", "--whisper_model", "base"], check=True)


## Optional: LLM Clinical Marker Panel

Manual API-cost experiment. This is not run by default. It sends only anonymous session IDs and transcript text, never audio, age, sex, or labels. Set `OPENAI_API_KEY` before running.


In [ ]:
# Optional manual run: LLM clinical marker panel.
# Dry run makes no API calls. Remove --dry_run for actual scoring.
# subprocess.run([sys.executable, "scripts/run_llm_clinical_marker_panel.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/llm_clinical_marker_panel", "--model", "gpt-4o-mini", "--max_sessions", "10", "--dry_run"], check=True)
# subprocess.run([sys.executable, "scripts/run_llm_clinical_marker_panel.py", "--data_dir", DATA_DIR, "--out_dir", "outputs/llm_clinical_marker_panel", "--model", "gpt-4o-mini"], check=True)


## Optional Slow Experiments

Full HuBERT layer sweep is intentionally not run by default. To run it manually after the default pipeline completes, uncomment and execute the cell below.


In [ ]:
# Optional: full HuBERT layer sweep, slow. Not run by default.
# subprocess.run([sys.executable, "scripts/run_hubert_layer_sweep.py", "--data_dir", DATA_DIR], check=True)
